# 1 — A sparse autoencoder, with an answer key

A language model stores more distinct concepts than it has dimensions. It gets away
with this by letting concepts share dimensions — a scheme called **superposition**.
The cost is that no single dimension means anything on its own.

A sparse autoencoder tries to undo that. It widens the activation into a much larger
space and insists only a few coordinates may be non-zero at once. The bet is that the
cheapest way to satisfy both demands is to give each coordinate one real concept.

Does that bet pay off? On a real model you can't tell — there's no ground truth to
check against. So here we build data whose true features we planted ourselves, and
grade the SAE against them.

**Run every cell top to bottom, then go back and turn the knobs flagged with 🔧.**

In [1]:
import numpy as np
import torch
import plotly.express as px
import plotly.graph_objects as go
from toydata import make_superposition_data
from sae import SAE, train_sae, feature_recovery, sae_metrics

N_TRUE, N_DIMS, N_LATENTS = 8, 5, 32   # 🔧 8 features crammed into 5 dimensions
SPARSITY = 0.05                        # 🔧 how often each feature fires
L1 = 1e-2                              # 🔧 the sparsity/accuracy exchange rate
STEPS = 15_000                         # 🔧 try 4_000 and watch recovery collapse

data = make_superposition_data(50_000, N_TRUE, N_DIMS, sparsity=SPARSITY, seed=0)
X = torch.tensor(data.X, dtype=torch.float32)
print(f"{N_TRUE} true features living in {N_DIMS} dimensions")
print(f"activations X: {tuple(X.shape)}")
print(f"on average {(data.coeffs > 0).sum(1).mean():.2f} features fire per sample")

8 true features living in 5 dimensions
activations X: (50000, 5)
on average 0.40 features fire per sample


## Why the features must interfere

Eight unit vectors in five dimensions cannot all be perpendicular — there isn't room.
So every feature partially overlaps others, and that overlap is the interference the
model tolerates in exchange for representing more things.

In [2]:
overlap = np.abs(data.true_features @ data.true_features.T)
np.fill_diagonal(overlap, 0)
print(f"largest |cosine| between two different true features: {overlap.max():.3f}")
print("(0.0 would mean perfectly separate; anything above 0 is interference)")

px.imshow(overlap, color_continuous_scale="Reds", zmin=0, zmax=1,
          labels=dict(x="feature", y="feature", color="|cos|"),
          title="Ground-truth features overlap because 8 > 5").show()

largest |cosine| between two different true features: 0.887
(0.0 would mean perfectly separate; anything above 0 is interference)


## The problem, concretely

Here is one activation vector. Five numbers. Which features produced it?

You can't read it off — and this is the easy case, where we *know* there are exactly
eight things to look for.

In [3]:
i = int(np.argmax((data.coeffs > 0).sum(1)))
print("activation x       :", np.round(data.X[i], 3))
print("features that fired:", np.where(data.coeffs[i] > 0)[0].tolist())
print("\nNothing about the five numbers announces which features they came from.")

activation x       : [-0.298 -0.393  0.092 -0.213 -0.331]
features that fired: [0, 1, 3, 5, 6]

Nothing about the five numbers announces which features they came from.


## First attempt: PCA

The obvious move is to find the data's principal directions. Watch *how* it fails —
it's more specific than "PCA is bad".

PCA is restricted to **orthogonal** directions and can return at most `n_dims` of them.
Some individual features do get a respectable match. The problem is that the matches
aren't *exclusive*: several different true features end up pointing at the same
component, so there's no way to read "which feature fired" off the result.

In [4]:
from sklearn.decomposition import PCA
pca = PCA(n_components=N_DIMS).fit(data.X)
comps = pca.components_ / np.linalg.norm(pca.components_, axis=1, keepdims=True)
truth = data.true_features / np.linalg.norm(data.true_features, axis=1, keepdims=True)
pca_cos = np.abs(truth @ comps.T)

best = pca_cos.argmax(1)
print(f"best PCA match per true feature: {np.round(pca_cos.max(1), 2)}")
print(f"matched at >0.9: {(pca_cos.max(1) > 0.9).sum()}/{N_TRUE}")
print(f"but those matches use only {len(set(best.tolist()))} distinct components: {best.tolist()}")
print()
print("Features collide onto the same component — that is the real failure.")

px.imshow(pca_cos, color_continuous_scale="Blues", zmin=0, zmax=1, aspect="auto",
          labels=dict(x="PCA component", y="true feature", color="|cos|"),
          title="PCA: several features collide onto the same component").show()

best PCA match per true feature: [0.79 0.94 0.92 0.93 0.92 0.59 0.77 0.75]
matched at >0.9: 4/8
but those matches use only 4 distinct components: [0, 0, 1, 0, 2, 1, 3, 0]

Features collide onto the same component — that is the real failure.


## Now the SAE

Same data, different constraint. Instead of 5 orthogonal directions we allow **32**
directions and require that only a couple are active at a time.

The loss is two terms in tension:

```
loss  =  ‖x − x̂‖²        reconstruct well  → wants many latents active
       + λ · ‖z‖₁         stay sparse       → wants few latents active
```

`λ` (`L1` above) sets the exchange rate between them. That single number is most of
what there is to tune.

In [5]:
sae = SAE(d_in=N_DIMS, n_latents=N_LATENTS, seed=0)
history = train_sae(sae, X, l1_coeff=L1, steps=STEPS, batch_size=1024, seed=0)

fig = go.Figure()
w = 200
smooth = lambda v: np.convolve(v, np.ones(w)/w, mode="valid")
fig.add_scatter(y=smooth(history["mse"]), name="reconstruction error")
fig.add_scatter(y=smooth(history["l0"]), name="L0 (latents active)", yaxis="y2")
fig.update_layout(title="Training: error falls while the code stays sparse",
                  xaxis_title="step", yaxis_title="MSE",
                  yaxis2=dict(title="L0", overlaying="y", side="right"))
fig.show()

## The money plot

Every true feature against every learned decoder direction. If the SAE worked, each
row has exactly one bright cell: that feature was found, and it lives in one latent.

A permuted identity matrix is what success looks like. (Order is arbitrary — latent 17
has no reason to be feature 0.)

In [6]:
rec = feature_recovery(sae, data.true_features)
m = sae_metrics(sae, X)

print(f"recovered {rec.n_recovered(0.9)}/{N_TRUE} features at cos > 0.9")
print(f"best cosine per true feature: {np.round(rec.best_cos, 3)}")
print(f"\nL0 {m.l0:.2f} active of {N_LATENTS} | "
      f"variance unexplained {m.fraction_variance_unexplained:.1%} | dead {m.n_dead}")

px.imshow(rec.cos_matrix, color_continuous_scale="RdBu", zmin=-1, zmax=1, aspect="auto",
          labels=dict(x="SAE latent", y="true feature", color="cos"),
          title="One bright cell per row = feature recovered").show()

recovered 8/8 features at cos > 0.9
best cosine per true feature: [0.939 0.986 0.943 0.966 0.96  0.953 0.909 0.927]

L0 2.21 active of 32 | variance unexplained 0.1% | dead 0


## The trade-off, drawn

`λ` is not a knob with a correct setting — it selects a point on a curve. Push it up and
the code gets sparser but reconstruction degrades; push it down and you reconstruct
beautifully with a code that means nothing.

Two things to watch, because they behave differently:

- **Error and dead latents move monotonically.** More sparsity pressure always costs
  reconstruction, and past a point it starts switching latents off permanently.
- **Recovery does not.** It stays high across a wide band and wobbles. So recovery
  alone is a bad thing to tune on — at λ=0.3 you can still recover every feature while
  having thrown away 11% of the variance and killed half your dictionary.

The balanced point here is λ=0.01: all features found, near-zero error, nothing dead.

*(~1 min: it trains six SAEs.)*

In [7]:
lambdas = [1e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0]
rows = []
for lam in lambdas:
    s = SAE(d_in=N_DIMS, n_latents=N_LATENTS, seed=0)
    train_sae(s, X, l1_coeff=lam, steps=STEPS, batch_size=1024, seed=0)
    mm, rr = sae_metrics(s, X), feature_recovery(s, data.true_features)
    rows.append((lam, mm.l0, mm.fraction_variance_unexplained, rr.n_recovered(0.9), mm.n_dead))
    print(f"λ={lam:<7g} L0={mm.l0:5.2f}  unexplained={mm.fraction_variance_unexplained:6.1%}"
          f"  recovered={rr.n_recovered(0.9)}/{N_TRUE}  dead={mm.n_dead}")

lam, l0s, fvus, recs, deads = zip(*rows)
fig = go.Figure()
fig.add_scatter(x=l0s, y=[f*100 for f in fvus], mode="markers+lines+text",
                text=[f"λ={l:g}" for l in lam], textposition="top center", name="error")
fig.update_layout(title="Sparser codes reconstruct worse — pick your point on this curve",
                  xaxis_title="L0  (latents active, lower = sparser)",
                  yaxis_title="variance unexplained (%)")
fig.show()

fig = go.Figure()
fig.add_bar(x=[f"{l:g}" for l in lam], y=recs, name=f"recovered of {N_TRUE}")
fig.add_bar(x=[f"{l:g}" for l in lam], y=deads, name="dead latents")
fig.update_layout(barmode="group", xaxis_title="λ",
                  title="Recovery holds up across a wide band; dead latents do not")
fig.show()

λ=0.001   L0= 3.35  unexplained=  0.0%  recovered=6/8  dead=4


λ=0.01    L0= 2.21  unexplained=  0.1%  recovered=8/8  dead=0


λ=0.03    L0= 1.65  unexplained=  0.4%  recovered=6/8  dead=2


λ=0.1     L0= 0.86  unexplained=  1.9%  recovered=8/8  dead=8


λ=0.3     L0= 0.43  unexplained= 11.2%  recovered=8/8  dead=15


λ=1       L0= 0.26  unexplained= 63.9%  recovered=6/8  dead=14


## The other way to be sparse: TopK

Instead of *penalising* density, you can simply forbid it — keep the k largest latents
and zero the rest. Sparsity becomes exact and there's no λ to tune.

The catch: you have to already know how sparse the truth is. Our data fires about
**0.4 features per sample**, so `k=1` is close to correct and recovers everything.
Set `k=5` and you have forced five latents to fire when only one real feature is
present — the SAE fills the surplus by splitting features across latents, and recovery
collapses.

Watch recovery fall as `k` rises above the true sparsity. With L1 you never had to
guess that number; the penalty found it.

In [8]:
for k in [1, 2, 3, 5, 10]:
    s = SAE(d_in=N_DIMS, n_latents=N_LATENTS, k=k, seed=0)
    train_sae(s, X, l1_coeff=0.0, steps=6_000, batch_size=1024, seed=0)
    mm, rr = sae_metrics(s, X), feature_recovery(s, data.true_features)
    print(f"k={k:<3d} L0={mm.l0:4.1f}  unexplained={mm.fraction_variance_unexplained:6.1%}"
          f"  recovered={rr.n_recovered(0.9)}/{N_TRUE}  dead={mm.n_dead}")

k=1   L0= 1.0  unexplained=  6.1%  recovered=8/8  dead=17


k=2   L0= 2.0  unexplained=  1.5%  recovered=7/8  dead=17


k=3   L0= 3.0  unexplained=  0.5%  recovered=6/8  dead=10


k=5   L0= 5.0  unexplained=  0.0%  recovered=1/8  dead=6


k=10  L0=10.0  unexplained=  0.0%  recovered=2/8  dead=3


## 🔧 Things worth breaking

Each of these changes a number you can now read off directly.

1. **`STEPS = 4_000`.** Recovery drops from 8/8 to about 4/8 — and the reconstruction
   error barely looks worse. Under-training doesn't announce itself; the SAE looks
   fine and quietly merges two features into one latent. This is the most common way
   to be fooled by an SAE.
2. **`SPARSITY = 0.2`.** Features now fire together constantly, so the SAE learns
   latents for *combinations*. Ground truth stops being recoverable.
3. **`N_LATENTS = 8`.** Exactly enough latents, no slack. Recovery gets much worse —
   overcompleteness isn't a luxury, it's how the SAE finds room to separate things.
4. **`N_DIMS = 8`** (with `N_TRUE = 8`). No superposition at all. Everything works
   trivially, which is the point: SAEs solve a problem that only exists when
   representations are crowded.
5. **`L1 = 1e-1`.** Watch dead latents climb. Too much pressure and most of the
   dictionary switches off permanently.

When you're done, open **`02_gpt2.ipynb`** — same `SAE` class, real GPT-2 activations,
and no answer key.